### Instalação das dependências

In [ ]:
!pip install -q -U keras-tuner
!pip install -q seaborn==0.13.2 matplotlib==3.10.1
!pip install -q astroNN
!pip install -q scikit-learn
!pip install -q gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 29.2 MB/s eta 0:00:00


In [ ]:
!gdown --id 1FAWD8ISlsQGuciE4-_8WHvH9HaLqFUhE -O galaxy10_subset.npz

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1FAWD8ISlsQGuciE4-_8WHvH9HaLqFUhE
From (redirected): https://drive.google.com/uc?id=1FAWD8ISlsQGuciE4-_8WHvH9HaLqFUhE&confirm=t&uuid=161479ec-d18d-4fa6-a2e2-e2eee06b2989
To: /content/galaxy10_subset.npz
100% 275M/275M [00:04<00:00, 58.5MB/s]


### Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras import utils, layers, models
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import EfficientNetV2M
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Rescaling
from astroNN.datasets import load_galaxy10
import collections
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import keras_tuner as kt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# Carregamento do Dataset (Escolher entre dataset inteiro ou 10%)

## Dataset inteiro

In [ ]:
images_orig, labels_orig = load_galaxy10()

## 10% do Dataset

In [ ]:
data = np.load("galaxy10_subset.npz")
images_orig = data["images"]
labels_orig = data["labels"]

# Experimento 1: Arquitetura Própria

## Experimento 1.1: Arquitetura base da CNN

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

Treino: (1238, 256, 256, 3) Validação: (265, 256, 256, 3) Teste: (266, 256, 256, 3)


### Inicialização do Modelo

In [ ]:
input_shape = (256, 256, 3)
model = models.Sequential([
    layers.Input(shape=input_shape),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(10, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

### Treinamento do Modelo

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=35,
    batch_size=32
)

Epoch 1/35
39/39 ━━━━━━━━━━━━━━━━━━━━ 197s 5s/step - accuracy: 0.1026 - loss: 2.5770 - val_accuracy: 0.1472 - val_loss: 2.2478
Epoch 2/35
39/39 ━━━━━━━━━━━━━━━━━━━━ 203s 5s/step - accuracy: 0.1333 - loss: 2.2800 - val_accuracy: 0.1472 - val_loss: 2.2499
Epoch 3/35
39/39 ━━━━━━━━━━━━━━━━━━━━ 186s 5s/step - accuracy: 0.1451 - loss: 2.2558 - val_accuracy: 0.1472 - val_loss: 2.2283
Epoch 4/35
39/39 ━━━━━━━━━━━━━━━━━━━━ 190s 5s/step - accuracy: 0.1677 - loss: 2.2310 - val_accuracy: 0.1472 - val_loss: 2.2151
Epoch 5/35
39/39 ━━━━━━━━━━━━━━━━━━━━ 193s 5s/step - accuracy: 0.1753 - loss: 2.2136 - val_accuracy: 0.1925 - val_loss: 2.1300
Epoch 6/35
39/39 ━━━━━━━━━━━━━━━━━━━━ 195s 5s/step - accuracy: 0.2412 - loss: 2.1004 - val_accuracy: 0.2189 - val_loss: 2.1038
Epoch 7/35
39/39 ━━━━━━━━━━━━━━━━━━━━ 196s 5s/step - accuracy: 0.2344 - loss: 2.0812 - val_accuracy: 0.2377 - val_loss: 2.0539
Epoch 8/35
39/39 ━━━━━━━━━━━━━━━━━━━━ 179s 5s/step - accuracy: 0.2957 - loss: 1.9196 - val_accuracy: 0.2528 - v

### Avaliação

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

# --- Gráfico da Acurácia ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

# --- Gráfico de Loss ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## Experimento 1.2: Arquitetura base da CNN com Oversampling

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Data Augmentation

#### Descobre qual a quantidade de imagens na maior classe

In [ ]:
counter = Counter(y_train)
max_count = max(counter.values())

print("Distribuição antes do augmentation:")
for cls, count in counter.items():
    print(f"Classe {cls}: {count}")

#### Configuração do gerador das imagens sintéticas

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

#### Gera as imagens para as classes minoritárias

In [ ]:
X_augmented = []
y_augmented = []

for cls, count in counter.items():
    n_to_add = max_count - count
    if n_to_add > 0:
        print(f"Classe {cls}: gerando {n_to_add} imagens extras para balancear")

        idx = np.where(y_train == cls)[0]
        X_cls = X_train[idx]
        y_cls = y_train[idx]

        gen = datagen.flow(
            X_cls, y_cls,
            batch_size=1,
            shuffle=True)

        for i in range(n_to_add):
            x_batch, y_batch = next(gen)
            X_augmented.append(x_batch[0])
            y_augmented.append(y_batch[0])

#### Concatena ao dataset original

In [ ]:
X_train_balanced = np.concatenate([X_train, np.array(X_augmented)])
y_train_balanced = np.concatenate([y_train, np.array(y_augmented).astype(y_train.dtype)])

#### Confere distribuição final

In [ ]:
new_counter = Counter(y_train_balanced)
print("\nDistribuição depois do augmentation:")
for cls, count in new_counter.items():
    print(f"Classe {cls}: {count}")

### Inicialização do Modelo

In [ ]:
input_shape = (256, 256, 3)
model = models.Sequential([
    layers.Input(shape=input_shape),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(10, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

### Treinamento do Modelo

In [ ]:
history = model.fit(
    X_train_balanced, y_train_balanced,
    validation_data=(X_val, y_val),
    epochs=35,
    batch_size=32
)

### Avaliação

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

# --- Gráfico da Acurácia ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

# --- Gráfico de Loss ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## Experimento 1.3: Arquitetura base da CNN com Oversampling e uso do Keras Tuner (Busca Ampla)

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Data Augmentation

#### Descobre qual a quantidade de imagens na maior classe

In [ ]:
counter = Counter(y_train)
max_count = max(counter.values())

print("Distribuição antes do augmentation:")
for cls, count in counter.items():
    print(f"Classe {cls}: {count}")

#### Configuração do gerador das imagens sintéticas

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

#### Gera as imagens para as classes minoritárias

In [ ]:
X_augmented = []
y_augmented = []

for cls, count in counter.items():
    n_to_add = max_count - count
    if n_to_add > 0:
        print(f"Classe {cls}: gerando {n_to_add} imagens extras para balancear")

        idx = np.where(y_train == cls)[0]
        X_cls = X_train[idx]
        y_cls = y_train[idx]

        gen = datagen.flow(
            X_cls, y_cls,
            batch_size=1,
            shuffle=True)

        for i in range(n_to_add):
            x_batch, y_batch = next(gen)
            X_augmented.append(x_batch[0])
            y_augmented.append(y_batch[0])

#### Concatena ao dataset original

In [ ]:
X_train_balanced = np.concatenate([X_train, np.array(X_augmented)])
y_train_balanced = np.concatenate([y_train, np.array(y_augmented).astype(y_train.dtype)])

#### Confere distribuição final

In [ ]:
new_counter = Counter(y_train_balanced)
print("\nDistribuição depois do augmentation:")
for cls, count in new_counter.items():
    print(f"Classe {cls}: {count}")

### Inicialização do modelo utilizando Keras Tuner

In [ ]:
def model_builder(hp):
    input_shape = (256, 256, 3)

    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))

    model.add(layers.Conv2D(
        filters=hp.Int('conv_1_filters', min_value=32, max_value=128, step=32),
        kernel_size=(3, 3),
        activation='relu'
    ))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(
        filters=hp.Int('conv_2_filters', min_value=64, max_value=256, step=64),
        kernel_size=(3, 3),
        activation='relu'
    ))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(
        filters=hp.Int('conv_3_filters', min_value=128, max_value=512, step=64),
        kernel_size=(3, 3),
        activation='relu'
    ))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Flatten())

    model.add(layers.Dense(
        units=hp.Int('dense_1_units', min_value=64, max_value=256, step=64),
        activation='relu'
    ))
    model.add(layers.Dropout(
        rate=hp.Float('dropout_1', min_value=0.3, max_value=0.7, step=0.1)
    ))

    model.add(layers.Dense(
        units=hp.Int('dense_2_units', min_value=32, max_value=128, step=32),
        activation='relu'
    ))
    model.add(layers.Dropout(
        rate=hp.Float('dropout_2', min_value=0.3, max_value=0.7, step=0.1)
    ))

    model.add(layers.Dense(10, activation='softmax'))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


#### Configuração do Hyperband e EarlyStopping

In [ ]:
tuner = kt.Hyperband(
    model_builder,
    objective='val_accuracy',
    max_epochs=25,
    factor=3,
    directory='TCC',
    project_name='tuner_base'
)

In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

#### Busca dos melhores hiperparâmetros

In [ ]:
tuner.search(
    X_train_balanced,
    y_train_balanced,
    epochs=50,
    validation_split=0.2,
    callbacks=[stop_early]
)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
Melhores hiperparâmetros:
Conv1 filters: {best_hps.get('conv_1_filters')}
Conv2 filters: {best_hps.get('conv_2_filters')}
Conv3 filters: {best_hps.get('conv_3_filters')}
Dense1 units: {best_hps.get('dense_1_units')}
Dense2 units: {best_hps.get('dense_2_units')}
Learning rate: {best_hps.get('learning_rate')}
""")

### Treinamento Final do Modelo

In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train_balanced,
    y_train_balanced,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[stop_early]
)

val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % best_epoch)

### Avaliação

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

# --- Gráfico da Acurácia ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

# --- Gráfico de Loss ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## Experimento 1.4: Arquitetura base da CNN com Oversampling e redução do espaço de busca do Keras Tuner

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Data Augmentation

#### Descobre qual a quantidade de imagens na maior classe

In [ ]:
counter = Counter(y_train)
max_count = max(counter.values())

print("Distribuição antes do augmentation:")
for cls, count in counter.items():
    print(f"Classe {cls}: {count}")

#### Configuração do gerador das imagens sintéticas

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

#### Gera as imagens para as classes minoritárias

In [ ]:
X_augmented = []
y_augmented = []

for cls, count in counter.items():
    n_to_add = max_count - count
    if n_to_add > 0:
        print(f"Classe {cls}: gerando {n_to_add} imagens extras para balancear")

        idx = np.where(y_train == cls)[0]
        X_cls = X_train[idx]
        y_cls = y_train[idx]

        gen = datagen.flow(
            X_cls, y_cls,
            batch_size=1,
            shuffle=True)

        for i in range(n_to_add):
            x_batch, y_batch = next(gen)
            X_augmented.append(x_batch[0])
            y_augmented.append(y_batch[0])

#### Concatena ao dataset original

In [ ]:
X_train_balanced = np.concatenate([X_train, np.array(X_augmented)])
y_train_balanced = np.concatenate([y_train, np.array(y_augmented).astype(y_train.dtype)])

#### Confere distribuição final

In [ ]:
new_counter = Counter(y_train_balanced)
print("\nDistribuição depois do augmentation:")
for cls, count in new_counter.items():
    print(f"Classe {cls}: {count}")

### Inicialização do modelo utilizando Keras Tuner

In [ ]:
def model_builder(hp):
    input_shape = (256, 256, 3)
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))

    model.add(layers.Conv2D(filters=hp.Choice('conv_1_filters', [32, 64]), kernel_size=(3,3), activation='relu'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(filters=hp.Choice('conv_2_filters', [64, 128]), kernel_size=(3,3), activation='relu'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(filters=hp.Choice('conv_3_filters', [128, 256]), kernel_size=(3,3), activation='relu'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Flatten())

    model.add(layers.Dense(units=hp.Choice('dense_1_units', [64, 128]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_1', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(units=hp.Choice('dense_2_units', [32, 64]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_2', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(10, activation='softmax'))

    hp_learning_rate = hp.Choice('learning_rate', [1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

#### Configuração do Hyperband e EarlyStopping

In [ ]:
tuner = kt.Hyperband(
    model_builder,
    objective='val_accuracy',
    max_epochs=25,
    factor=3,
    directory='TCC',
    project_name='tuner_otimizado'
)

In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

#### Busca dos melhores hiperparâmetros

In [ ]:
tuner.search(
    X_train_balanced,
    y_train_balanced,
    epochs=25,
    validation_data=(X_val, y_val),
    callbacks=[stop_early]
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
Melhores hiperparâmetros:
Conv1 filters: {best_hps.get('conv_1_filters')}
Conv2 filters: {best_hps.get('conv_2_filters')}
Conv3 filters: {best_hps.get('conv_3_filters')}
Dense1 units: {best_hps.get('dense_1_units')}
Dense2 units: {best_hps.get('dense_2_units')}
Learning rate: {best_hps.get('learning_rate')}
""")

### Treinamento Final do Modelo

In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train_balanced,
    y_train_balanced,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[stop_early]
)
val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % best_epoch)

### Avaliação

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

# --- Gráfico da Acurácia ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

# --- Gráfico de Loss ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## Experimento 1.5: Expansão da Arquitetura



### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Data Augmentation

#### Descobre qual a quantidade de imagens na maior classe

In [ ]:
counter = Counter(y_train)
max_count = max(counter.values())

print("Distribuição antes do augmentation:")
for cls, count in counter.items():
    print(f"Classe {cls}: {count}")

#### Configuração do gerador das imagens sintéticas

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

#### Gera as imagens para as classes minoritárias

In [ ]:
X_augmented = []
y_augmented = []

for cls, count in counter.items():
    n_to_add = max_count - count
    if n_to_add > 0:
        print(f"Classe {cls}: gerando {n_to_add} imagens extras para balancear")

        idx = np.where(y_train == cls)[0]
        X_cls = X_train[idx]
        y_cls = y_train[idx]

        gen = datagen.flow(
            X_cls, y_cls,
            batch_size=1,
            shuffle=True)

        for i in range(n_to_add):
            x_batch, y_batch = next(gen)
            X_augmented.append(x_batch[0])
            y_augmented.append(y_batch[0])

#### Concatena ao dataset original

In [ ]:
X_train_balanced = np.concatenate([X_train, np.array(X_augmented)])
y_train_balanced = np.concatenate([y_train, np.array(y_augmented).astype(y_train.dtype)])

#### Confere distribuição final

In [ ]:
new_counter = Counter(y_train_balanced)
print("\nDistribuição depois do augmentation:")
for cls, count in new_counter.items():
    print(f"Classe {cls}: {count}")

### Inicialização do modelo utilizando Keras Tuner

In [ ]:
def model_builder(hp):
    input_shape = (256, 256, 3)
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))

    model.add(layers.Conv2D(filters=hp.Choice('conv_1_filters', [32, 64]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_2_filters', [64, 128]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_3_filters', [128, 256]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_4_filters', [256, 512]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dense(units=hp.Choice('dense_1_units', [128, 256]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_1', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(units=hp.Choice('dense_2_units', [64, 128]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_2', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(10, activation='softmax'))

    hp_learning_rate = hp.Choice('learning_rate', [1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

#### Configuração do Hyperband e EarlyStopping

In [ ]:
tuner = kt.Hyperband(
    model_builder,
    objective='val_accuracy',
    max_epochs=25,
    factor=3,
    directory='TCC',
    project_name='tuner_mais_camadas'
)

In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=7, restore_best_weights=True
)

#### Busca dos melhores hiperparâmetros

In [ ]:
tuner.search(
    X_train_balanced,
    y_train_balanced,
    epochs=25,
    validation_data=(X_val, y_val),
    callbacks=[stop_early]
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
Melhores hiperparâmetros:
Conv1 filters: {best_hps.get('conv_1_filters')}
Conv2 filters: {best_hps.get('conv_2_filters')}
Conv3 filters: {best_hps.get('conv_3_filters')}
Conv4 filters: {best_hps.get('conv_4_filters')}
Dense1 units: {best_hps.get('dense_1_units')}
Dense2 units: {best_hps.get('dense_2_units')}
Learning rate: {best_hps.get('learning_rate')}
""")

### Treinamento Final do Modelo

In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train_balanced,
    y_train_balanced,
    epochs=30,
    validation_data=(X_val, y_val)
)
val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % best_epoch)

### Avaliação

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

# --- Gráfico da Acurácia ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

# --- Gráfico de Loss ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

##  Experimento 1.6: CNN Expandida com Oversampling

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Inicialização do modelo utilizando Keras Tuner

In [ ]:
def model_builder(hp):
    input_shape = (256, 256, 3)
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))

    model.add(layers.Conv2D(filters=hp.Choice('conv_1_filters', [32, 64]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_2_filters', [64, 128]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_3_filters', [128, 256]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_4_filters', [256, 512]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dense(units=hp.Choice('dense_1_units', [128, 256]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_1', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(units=hp.Choice('dense_2_units', [64, 128]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_2', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(10, activation='softmax'))

    hp_learning_rate = hp.Choice('learning_rate', [1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

#### Configuração do Hyperband e Early Stopping

In [ ]:
tuner = kt.Hyperband(
    model_builder,
    objective='val_accuracy',
    max_epochs=25,
    factor=3,
    directory='TCC',
    project_name='tuner_sem_data_augmentation'
)

In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=7, restore_best_weights=True
)

#### Busca dos melhores hiperparâmetros

In [ ]:
tuner.search(
    X_train,
    y_train,
    epochs=25,
    validation_data=(X_val, y_val),
    callbacks=[stop_early]
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
Melhores hiperparâmetros:
Conv1 filters: {best_hps.get('conv_1_filters')}
Conv2 filters: {best_hps.get('conv_2_filters')}
Conv3 filters: {best_hps.get('conv_3_filters')}
Conv4 filters: {best_hps.get('conv_4_filters')}
Dense1 units: {best_hps.get('dense_1_units')}
Dense2 units: {best_hps.get('dense_2_units')}
Learning rate: {best_hps.get('learning_rate')}
""")

### Treinamento Final do Modelo

In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    validation_data=(X_val, y_val)
)
val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % best_epoch)

### Avaliação

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

# --- Gráfico da Acurácia ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

# --- Gráfico de Loss ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## Experimento 1.7: CNN Expandida com Class Weights

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Ajuste de Peso das Classes

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))
print(class_weights)

### Inicialização do modelo utilizando Keras Tuner

In [ ]:
def model_builder(hp):
    input_shape = (256 ,256, 3)
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))

    model.add(layers.Conv2D(filters=hp.Choice('conv_1_filters', [32, 64]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_2_filters', [64, 128]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_3_filters', [128, 256]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_4_filters', [256, 512]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dense(units=hp.Choice('dense_1_units', [128, 256]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_1', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(units=hp.Choice('dense_2_units', [64, 128]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_2', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(10, activation='softmax'))

    hp_learning_rate = hp.Choice('learning_rate', [1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

#### Configuração do Hyperband e Early Stopping

In [ ]:
tuner = kt.Hyperband(
    model_builder,
    objective='val_accuracy',
    max_epochs=25,
    factor=3,
    directory='TCC',
    project_name='tuner_com_peso_das_classes'
)

In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=7, restore_best_weights=True
)

#### Busca dos melhores hiperparâmetros

In [ ]:
tuner.search(
    X_train,
    y_train,
    epochs=25,
    validation_data=(X_val, y_val),
    callbacks=[stop_early]
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
Melhores hiperparâmetros:
Conv1 filters: {best_hps.get('conv_1_filters')}
Conv2 filters: {best_hps.get('conv_2_filters')}
Conv3 filters: {best_hps.get('conv_3_filters')}
Conv4 filters: {best_hps.get('conv_4_filters')}
Dense1 units: {best_hps.get('dense_1_units')}
Dense2 units: {best_hps.get('dense_2_units')}
Learning rate: {best_hps.get('learning_rate')}
""")

### Treinamento Final do Modelo

In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    class_weight = class_weights,
    validation_data=(X_val, y_val)
)
val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % best_epoch)

### Avaliação

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

# --- Gráfico da Acurácia ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

# --- Gráfico de Loss ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## Experimento 1.8: CNN Expandida com Data Augmentation Seletivo

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Data Augmentation para as "Piores" Classes

In [ ]:
counter = Counter(y_train)
max_count = max(counter.values())

print("Distribuição antes do augmentation:")
for cls, count in counter.items():
    print(f"Classe {cls}: {count}")

#### Seleção das classes com pior acurácia

In [ ]:
CLASSES_TO_AUGMENT = [0, 4, 7, 1]
N_IMAGES_TO_ADD_PER_CLASS = 500

#### Configuração do gerador das imagens sintéticas

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

#### Gera as imagens para as classes selecionadas

In [ ]:
X_augmented = []
y_augmented = []

for cls in CLASSES_TO_AUGMENT:
    n_to_add = N_IMAGES_TO_ADD_PER_CLASS
    print(f"Classe {cls}: gerando {n_to_add} imagens extras para balancear")


    idx = np.where(y_train == cls)[0]
    X_cls = X_train[idx]
    y_cls = y_train[idx]


    gen = datagen.flow(
        X_cls, y_cls,
        batch_size=1,
        shuffle=True
    )

    for i in range(n_to_add):
        x_batch, y_batch = next(gen)
        X_augmented.append(x_batch[0])
        y_augmented.append(y_batch[0])

#### Concatena ao dataset original

In [ ]:
X_train_balanced = np.concatenate([X_train, np.array(X_augmented)])
y_train_balanced = np.concatenate([y_train, np.array(y_augmented).astype(y_train.dtype)])

#### Confere distribuição final

In [ ]:
new_counter = Counter(y_train_balanced)
print("\nDistribuição depois do augmentation:")
for cls, count in new_counter.items():
    print(f"Classe {cls}: {count}")

### Inicialização do modelo utilizando Keras Tuner

In [ ]:
def model_builder(hp):
    input_shape = (256, 256, 3)
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))

    model.add(layers.Conv2D(filters=hp.Choice('conv_1_filters', [32, 64]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_2_filters', [64, 128]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_3_filters', [128, 256]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_4_filters', [256, 512]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dense(units=hp.Choice('dense_1_units', [128, 256]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_1', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(units=hp.Choice('dense_2_units', [64, 128]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_2', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(10, activation='softmax'))

    hp_learning_rate = hp.Choice('learning_rate', [1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

#### Configuração do Hyperband e EarlyStopping

In [ ]:
tuner = kt.Hyperband(
    model_builder,
    objective='val_accuracy',
    max_epochs=25,
    factor=3,
    directory='TCC',
    project_name='tuner_com_data_augmentation_seletivo'
)

In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=7, restore_best_weights=True
)

#### Busca dos melhores hiperparâmetros

In [ ]:
tuner.search(
    X_train_balanced,
    y_train_balanced,
    epochs=25,
    validation_data=(X_val, y_val),
    callbacks=[stop_early]
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
Melhores hiperparâmetros:
Conv1 filters: {best_hps.get('conv_1_filters')}
Conv2 filters: {best_hps.get('conv_2_filters')}
Conv3 filters: {best_hps.get('conv_3_filters')}
Conv4 filters: {best_hps.get('conv_4_filters')}
Dense1 units: {best_hps.get('dense_1_units')}
Dense2 units: {best_hps.get('dense_2_units')}
Learning rate: {best_hps.get('learning_rate')}
""")

### Treinamento Final do Modelo

In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train_balanced,
    y_train_balanced,
    epochs=30,
    validation_data=(X_val, y_val)
)
val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % best_epoch)

### Avaliação

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

# --- Gráfico da Acurácia ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

# --- Gráfico de Loss ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

##  Experimento 1.9: CNN Expandida e Data Augmentation Seletivo com novos parâmetros

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Data Augmentation para as "Piores" Classes

In [ ]:
counter = Counter(y_train)
max_count = max(counter.values())

print("Distribuição antes do augmentation:")
for cls, count in counter.items():
    print(f"Classe {cls}: {count}")

#### Seleção das classes com pior acurácia

In [ ]:
CLASSES_TO_AUGMENT = [0, 4, 7, 1]
N_IMAGES_TO_ADD_PER_CLASS = 500

#### Configuração do gerador das imagens sintéticas

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'

)

#### Gera as imagens para as classes selecionadas

In [ ]:
X_augmented = []
y_augmented = []

for cls in CLASSES_TO_AUGMENT:
    n_to_add = N_IMAGES_TO_ADD_PER_CLASS
    print(f"Classe {cls}: gerando {n_to_add} imagens extras para balancear")


    idx = np.where(y_train == cls)[0]
    X_cls = X_train[idx]
    y_cls = y_train[idx]


    gen = datagen.flow(
        X_cls, y_cls,
        batch_size=1,
        shuffle=True
    )

    for i in range(n_to_add):
        x_batch, y_batch = next(gen)
        X_augmented.append(x_batch[0])
        y_augmented.append(y_batch[0])

#### Concatena ao dataset original

In [ ]:
X_train_balanced = np.concatenate([X_train, np.array(X_augmented)])
y_train_balanced = np.concatenate([y_train, np.array(y_augmented).astype(y_train.dtype)])


#### Confere distribuição final


In [ ]:
new_counter = Counter(y_train_balanced)
print("\nDistribuição depois do augmentation:")
for cls, count in new_counter.items():
    print(f"Classe {cls}: {count}")

### Inicialização do modelo utilizando Keras Tuner

In [ ]:
def model_builder(hp):
    input_shape = (256, 256, 3)
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))

    model.add(layers.Conv2D(filters=hp.Choice('conv_1_filters', [32, 64]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_2_filters', [64, 128]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_3_filters', [128, 256]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.Conv2D(filters=hp.Choice('conv_4_filters', [256, 512]), kernel_size=(3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.2))

    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dense(units=hp.Choice('dense_1_units', [128, 256]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_1', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(units=hp.Choice('dense_2_units', [64, 128]), activation='relu'))
    model.add(layers.Dropout(rate=hp.Float('dropout_2', 0.3, 0.5, step=0.1)))

    model.add(layers.Dense(10, activation='softmax'))

    hp_learning_rate = hp.Choice('learning_rate', [1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

#### Configuração do Hyperband e EarlyStopping

In [ ]:
tuner = kt.Hyperband(
    model_builder,
    objective='val_accuracy',
    max_epochs=25,
    factor=3,
    directory='TCC',
    project_name='tuner_com_data_augmentation_seletivo_novos_parametros'
)

In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=7, restore_best_weights=True
)

#### Busca dos melhores hiperparâmetros

In [ ]:
tuner.search(
    X_train_balanced,
    y_train_balanced,
    epochs=25,
    validation_data=(X_val, y_val),
    callbacks=[stop_early]
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
Melhores hiperparâmetros:
Conv1 filters: {best_hps.get('conv_1_filters')}
Conv2 filters: {best_hps.get('conv_2_filters')}
Conv3 filters: {best_hps.get('conv_3_filters')}
Conv4 filters: {best_hps.get('conv_4_filters')}
Dense1 units: {best_hps.get('dense_1_units')}
Dense2 units: {best_hps.get('dense_2_units')}
Learning rate: {best_hps.get('learning_rate')}
""")

### Treinamento Final do Modelo

In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train_balanced,
    y_train_balanced,
    epochs=30,
    validation_data=(X_val, y_val)
)
val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % best_epoch)

### Avaliação

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

# --- Gráfico da Acurácia ---
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

# --- Gráfico de Loss ---
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Experimento 2: ResNet-50


## Experimento 2.1: Baseline ResNet-50 com Oversampling

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [ ]:
def balanced_data_generator(X, y, batch_size):
    class_counts = Counter(y)
    class_indices = {cls: np.where(y == cls)[0] for cls in class_counts.keys()}
    num_classes = len(class_counts)

    while True:
        X_batch = []
        y_batch = []

        samples_per_class = max(1, batch_size // num_classes)

        for _ in range(samples_per_class):
            for cls in sorted(class_counts.keys()):
                idx = np.random.choice(class_indices[cls])

                image = X[idx]

                augmented_image = datagen.random_transform(image)

                processed_image = tf.keras.applications.resnet50.preprocess_input(augmented_image)

                X_batch.append(processed_image)
                y_batch.append(y[idx])

        yield (np.array(X_batch), np.array(y_batch))

### Inicialização do Modelo

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(256, 256, 3))

x = layers.Resizing(224, 224)(inputs)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
def validation_test_generator(X, y, batch_size):
    """Gera lotes para validação/teste com pré-processamento em tempo real."""
    num_samples = len(X)
    while True:
        for offset in range(0, num_samples, batch_size):
            X_batch = X[offset:offset+batch_size]
            y_batch = y[offset:offset+batch_size]

            X_processed_batch = tf.keras.applications.resnet50.preprocess_input(X_batch)

            yield (X_processed_batch, y_batch)

In [ ]:
BATCH_SIZE = 32

train_generator = balanced_data_generator(X_train, y_train, BATCH_SIZE)
val_generator = validation_test_generator(X_val, y_val, BATCH_SIZE)
test_generator = validation_test_generator(X_test, y_test, BATCH_SIZE)

In [ ]:
steps_per_epoch = math.ceil(len(X_train) / BATCH_SIZE)
validation_steps = math.ceil(len(X_val) / BATCH_SIZE)
test_steps = math.ceil(len(X_test) / BATCH_SIZE)

In [ ]:
checkpoint = ModelCheckpoint('best_model_resnet50.keras', save_best_only=True, monitor='val_accuracy', mode='max')
early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

### Treinamento Final do Modelo

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=val_generator,
    validation_steps=validation_steps,
    callbacks=[checkpoint, early_stopping]
)

In [ ]:
best_model = load_model('best_model_resnet50.keras')

### Avaliação

In [ ]:
X_test_processed = tf.keras.applications.resnet50.preprocess_input(X_test)
test_loss, test_acc = model.evaluate(X_test_processed, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = best_model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## Experimento 2.2: ResNet-50 com Fine-Tuning a partir do bloco conv5

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)


In [ ]:
def balanced_data_generator(X, y, batch_size):
    class_counts = Counter(y)
    class_indices = {cls: np.where(y == cls)[0] for cls in class_counts.keys()}
    num_classes = len(class_counts)

    while True:
        X_batch = []
        y_batch = []

        samples_per_class = max(1, batch_size // num_classes)

        for _ in range(samples_per_class):
            for cls in sorted(class_counts.keys()):
                idx = np.random.choice(class_indices[cls])

                image = X[idx]

                augmented_image = datagen.random_transform(image)

                processed_image = tf.keras.applications.resnet50.preprocess_input(augmented_image)

                X_batch.append(processed_image)
                y_batch.append(y[idx])

        yield (np.array(X_batch), np.array(y_batch))

### Inicialização do Modelo

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(256, 256, 3))

x = layers.Resizing(224, 224)(inputs)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
def validation_test_generator(X, y, batch_size):
    """Gera lotes para validação/teste com pré-processamento em tempo real."""
    num_samples = len(X)
    while True:
        for offset in range(0, num_samples, batch_size):
            X_batch = X[offset:offset+batch_size]
            y_batch = y[offset:offset+batch_size]

            X_processed_batch = tf.keras.applications.resnet50.preprocess_input(X_batch)

            yield (X_processed_batch, y_batch)

In [ ]:
BATCH_SIZE = 32

train_generator = balanced_data_generator(X_train, y_train, BATCH_SIZE)
val_generator = validation_test_generator(X_val, y_val, BATCH_SIZE)
test_generator = validation_test_generator(X_test, y_test, BATCH_SIZE)

In [ ]:
steps_per_epoch = math.ceil(len(X_train) / BATCH_SIZE)
validation_steps = math.ceil(len(X_val) / BATCH_SIZE)
test_steps = math.ceil(len(X_test) / BATCH_SIZE)

In [ ]:
checkpoint = ModelCheckpoint('best_model_resnet50_exp2.keras', save_best_only=True, monitor='val_accuracy', mode='max')
early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

### Treinamento do Modelo

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=val_generator,
    validation_steps=validation_steps,
    callbacks=[checkpoint, early_stopping]
)

In [ ]:
best_model = load_model('best_model_resnet50_exp2.keras')

In [ ]:
loss, accuracy = best_model.evaluate(test_generator, steps=test_steps)
print(f"Acurácia no teste: {accuracy:.2%}")

### Fine-Tuning

In [ ]:
try:
    model_for_finetuning = load_model('best_model_resnet50_exp2.keras')
    print("Modelo carregado com sucesso para fine-tuning.")

    base_model_finetune = model_for_finetuning.get_layer('resnet50')

except Exception as e:
    print(f"Erro ao carregar o modelo 'best_model_resnet50_exp2.keras': {e}")
    print("Verifique se o arquivo foi salvo corretamente na fase anterior.")
    base_model_finetune = model.get_layer('resnet50')

In [ ]:
base_model_finetune.trainable = True

fine_tune_at_layer_name = 'conv5_block1_out'

set_trainable = False
for layer in base_model_finetune.layers:
    if layer.name == fine_tune_at_layer_name:
        set_trainable = True
    if set_trainable:
        layer.trainable = True
    else:
        layer.trainable = False

print(f"\nNúmero total de camadas no base_model: {len(base_model_finetune.layers)}")
print(f"Número de camadas treináveis no base_model após descongelar: {len(base_model_finetune.trainable_weights) // 2}")

In [ ]:
model_for_finetuning.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_for_finetuning.summary()

### Treinamento Final do Modelo

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
print("\nContinuando o treinamento (Fine-Tuning)...")

checkpoint_ft = ModelCheckpoint('fine_tuned_model_exp2.keras', save_best_only=True, monitor='val_accuracy', mode='max')
early_stopping_ft = EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True)

reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', factor=0.2,
                            patience=3,
                            min_lr=1e-7,
                            verbose=1)

fine_tune_epochs = 25
total_epochs = 20 + fine_tune_epochs

history_fine = model_for_finetuning.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1,
    validation_data=val_generator,
    validation_steps=validation_steps,
    callbacks=[checkpoint_ft, early_stopping_ft,reduce_lr]
)


In [ ]:
best_model = load_model('fine_tuned_model_exp2.keras')

### Avaliação

In [ ]:
X_test_processed = tf.keras.applications.resnet50.preprocess_input(X_test)
loss, accuracy = best_model.evaluate(X_test_processed, y_test)
print(f"Acurácia final no conjunto de teste: {accuracy:.4f}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

acc += history_fine.history['accuracy']
val_acc += history_fine.history['val_accuracy']
loss += history_fine.history['loss']
val_loss += history_fine.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = best_model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## Experimento 2.3: ResNet-50 com Fine-Tuning a partir do bloco Conv4

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [ ]:
def balanced_data_generator(X, y, batch_size):
    class_counts = Counter(y)
    class_indices = {cls: np.where(y == cls)[0] for cls in class_counts.keys()}
    num_classes = len(class_counts)

    while True:
        X_batch = []
        y_batch = []

        samples_per_class = max(1, batch_size // num_classes)

        for _ in range(samples_per_class):
            for cls in sorted(class_counts.keys()):
                idx = np.random.choice(class_indices[cls])

                image = X[idx]

                augmented_image = datagen.random_transform(image)

                processed_image = tf.keras.applications.resnet50.preprocess_input(augmented_image)

                X_batch.append(processed_image)
                y_batch.append(y[idx])

        yield (np.array(X_batch), np.array(y_batch))

### Inicialização do Modelo

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(256, 256, 3))

x = layers.Resizing(224, 224)(inputs)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
def validation_test_generator(X, y, batch_size):
    """Gera lotes para validação/teste com pré-processamento em tempo real."""
    num_samples = len(X)
    while True:
        for offset in range(0, num_samples, batch_size):
            X_batch = X[offset:offset+batch_size]
            y_batch = y[offset:offset+batch_size]

            X_processed_batch = tf.keras.applications.resnet50.preprocess_input(X_batch)

            yield (X_processed_batch, y_batch)

In [ ]:
BATCH_SIZE = 32

train_generator = balanced_data_generator(X_train, y_train, BATCH_SIZE)
val_generator = validation_test_generator(X_val, y_val, BATCH_SIZE)
test_generator = validation_test_generator(X_test, y_test, BATCH_SIZE)

In [ ]:
steps_per_epoch = math.ceil(len(X_train) / BATCH_SIZE)
validation_steps = math.ceil(len(X_val) / BATCH_SIZE)
test_steps = math.ceil(len(X_test) / BATCH_SIZE)

In [ ]:
checkpoint = ModelCheckpoint('best_model_resnet50_conv4.keras', save_best_only=True, monitor='val_accuracy', mode='max')
early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

### Treinamento do Modelo

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=val_generator,
    validation_steps=validation_steps,
    callbacks=[checkpoint, early_stopping]
)

In [ ]:
best_model = load_model('best_model_resnet50_conv4.keras')

In [ ]:
loss, accuracy = best_model.evaluate(test_generator, steps=test_steps)
print(f"Acurácia no teste: {accuracy:.2%}")

### Fine-Tuning

In [ ]:
try:
    model_for_finetuning = load_model('best_model_resnet50_conv4.keras')
    base_model_finetune = model_for_finetuning.get_layer('resnet50')

except Exception as e:
    print(f"Erro ao carregar o modelo 'best_model_resnet50_conv4.keras': {e}")
    print("Verifique se o arquivo foi salvo corretamente na fase anterior.")
    base_model_finetune = model.get_layer('resnet50')

In [ ]:
base_model_finetune.trainable = True

fine_tune_at_layer_name = 'conv4_block1_out'

set_trainable = False
for layer in base_model_finetune.layers:
    if layer.name == fine_tune_at_layer_name:
        set_trainable = True
    if set_trainable:
        layer.trainable = True
    else:
        layer.trainable = False

print(f"\nNúmero total de camadas no base_model: {len(base_model_finetune.layers)}")
print(f"Número de camadas treináveis no base_model após descongelar: {len(base_model_finetune.trainable_weights) // 2}")

In [ ]:
model_for_finetuning.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_for_finetuning.summary()

### Treinamento Final do Modelo

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
checkpoint_ft = ModelCheckpoint('fine_tuned_model_conv4.keras', save_best_only=True, monitor='val_accuracy', mode='max')
early_stopping_ft = EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True)

reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', factor=0.2,
                            patience=3,
                            min_lr=1e-7,
                            verbose=1)

epochs_run_in_phase_1 = len(history.epoch)
fine_tune_epochs = 25

total_epochs = epochs_run_in_phase_1 + fine_tune_epochs
initial_epoch = epochs_run_in_phase_1

history_fine = model_for_finetuning.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=total_epochs,
    initial_epoch=initial_epoch,
    validation_data=val_generator,
    validation_steps=validation_steps,
    callbacks=[checkpoint_ft, early_stopping_ft,reduce_lr]
)


In [ ]:
best_model = load_model('fine_tuned_model_conv4.keras')

### Avaliação

In [ ]:
X_test_processed = tf.keras.applications.resnet50.preprocess_input(X_test)
test_loss, test_acc = model.evaluate(X_test_processed, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

acc += history_fine.history['accuracy']
val_acc += history_fine.history['val_accuracy']
loss += history_fine.history['loss']
val_loss += history_fine.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = best_model.predict(X_test_processed)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## Experimento 2.4: ResNet-50 com Fine-Tuning a partir do bloco Conv4 e ajuste nos parâmetros do Data Augmentation

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=90,
    width_shift_range=0.20,
    height_shift_range=0.20,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

In [ ]:
def balanced_data_generator(X, y, batch_size):
    class_counts = Counter(y)
    class_indices = {cls: np.where(y == cls)[0] for cls in class_counts.keys()}
    num_classes = len(class_counts)

    while True:
        X_batch = []
        y_batch = []

        samples_per_class = max(1, batch_size // num_classes)

        for _ in range(samples_per_class):
            for cls in sorted(class_counts.keys()):
                idx = np.random.choice(class_indices[cls])

                image = X[idx]

                augmented_image = datagen.random_transform(image)

                processed_image = tf.keras.applications.resnet50.preprocess_input(augmented_image)

                X_batch.append(processed_image)
                y_batch.append(y[idx])

        yield (np.array(X_batch), np.array(y_batch))

### Inicialização do Modelo

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)


base_model.trainable = False


inputs = layers.Input(shape=(256, 256, 3))


x = layers.Resizing(224, 224)(inputs)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
def validation_test_generator(X, y, batch_size):
    """Gera lotes para validação/teste com pré-processamento em tempo real."""
    num_samples = len(X)
    while True:
        for offset in range(0, num_samples, batch_size):
            X_batch = X[offset:offset+batch_size]
            y_batch = y[offset:offset+batch_size]

            X_processed_batch = tf.keras.applications.resnet50.preprocess_input(X_batch)

            yield (X_processed_batch, y_batch)

In [ ]:
X_val_processed = tf.keras.applications.resnet50.preprocess_input(X_val)

In [ ]:
BATCH_SIZE = 32

train_generator = balanced_data_generator(X_train, y_train, BATCH_SIZE)

In [ ]:
steps_per_epoch = math.ceil(len(X_train) / BATCH_SIZE)
validation_steps = math.ceil(len(X_val) / BATCH_SIZE)
test_steps = math.ceil(len(X_test) / BATCH_SIZE)

In [ ]:
checkpoint = ModelCheckpoint('best_model_resnet50_conv4_augmentation_90.keras', save_best_only=True, monitor='val_accuracy', mode='max')
early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

### Treinamento do Modelo

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=(X_val_processed, y_val),
    callbacks=[checkpoint, early_stopping]
)

In [ ]:
best_model = load_model('best_model_resnet50_conv4_augmentation_90.keras')

In [ ]:
X_test_processed = tf.keras.applications.resnet50.preprocess_input(X_test)
loss, accuracy = best_model.evaluate(X_test_processed, y_test)
print(f"Acurácia no teste: {accuracy:.2%}")

### Fine-Tuning

In [ ]:
try:
    model_for_finetuning = load_model('best_model_resnet50_conv4_augmentation_90.keras')
    print("Modelo carregado com sucesso para fine-tuning.")
    base_model_finetune = model_for_finetuning.get_layer('resnet50')

except Exception as e:
    print(f"Erro ao carregar o modelo 'best_model_resnet50_conv4_augmentation_90.keras': {e}")
    print("Verifique se o arquivo foi salvo corretamente na fase anterior.")

    base_model_finetune = model.get_layer('resnet50')

In [ ]:
base_model_finetune.trainable = True

fine_tune_at_layer_name = 'conv4_block1_out'
set_trainable = False
for layer in base_model_finetune.layers:
    if layer.name == fine_tune_at_layer_name:
        set_trainable = True
    if set_trainable:
        layer.trainable = True

    else:
        layer.trainable = False

print(f"\nNúmero total de camadas no base_model: {len(base_model_finetune.layers)}")
print(f"Número de camadas treináveis no base_model após descongelar: {len(base_model_finetune.trainable_weights) // 2}")

In [ ]:
model_for_finetuning.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_for_finetuning.summary()

### Treinamento Final do Modelo

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

print("\nContinuando o treinamento (Fine-Tuning)...")

checkpoint_ft = ModelCheckpoint('fine_tuned_model_conv4_augmentation_90.keras', save_best_only=True, monitor='val_accuracy', mode='max')

early_stopping_ft = EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True)

reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', factor=0.2,
                            patience=3,
                            min_lr=1e-7,
                            verbose=1)

epochs_run_in_phase_1 = len(history.epoch)
fine_tune_epochs = 25

total_epochs = epochs_run_in_phase_1 + fine_tune_epochs
initial_epoch = epochs_run_in_phase_1

history_fine = model_for_finetuning.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=total_epochs,
    initial_epoch=initial_epoch,
    validation_data=(X_val_processed, y_val),
    callbacks=[checkpoint_ft, early_stopping_ft,reduce_lr]
)


In [ ]:
best_model = load_model('fine_tuned_model_conv4_augmentation_90.keras')

### Avaliação

In [ ]:
X_test_processed = tf.keras.applications.resnet50.preprocess_input(X_test)
test_loss, test_acc = best_model.evaluate(X_test_processed, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']


acc += history_fine.history['accuracy']
val_acc += history_fine.history['val_accuracy']
loss += history_fine.history['loss']
val_loss += history_fine.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = best_model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## Experimento 2.5: ResNet-50 com Fine-Tuning a partir do bloco Conv3

### Pré-processamento e Normalização de Dados

In [ ]:
images = images_orig.copy()
labels = labels_orig.copy()
images = images.astype('float32') / 255.0

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]


classes, counts = np.unique(labels, return_counts=True)


class_labels = [class_names[i] for i in classes]

### Treinamento
Separação de 70% treino, validação e teste (15% cada)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=42, stratify=labels)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Treino:", X_train.shape, "Validação:", X_val.shape, "Teste:", X_test.shape)

### Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=90,
    width_shift_range=0.20,
    height_shift_range=0.20,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

In [ ]:
def balanced_data_generator(X, y, batch_size):
    class_counts = Counter(y)
    class_indices = {cls: np.where(y == cls)[0] for cls in class_counts.keys()}
    num_classes = len(class_counts)

    while True:
        X_batch = []
        y_batch = []

        samples_per_class = max(1, batch_size // num_classes)

        for _ in range(samples_per_class):
            for cls in sorted(class_counts.keys()):
                idx = np.random.choice(class_indices[cls])

                image = X[idx]

                augmented_image = datagen.random_transform(image)

                processed_image = tf.keras.applications.resnet50.preprocess_input(augmented_image)

                X_batch.append(processed_image)
                y_batch.append(y[idx])

        yield (np.array(X_batch), np.array(y_batch))

### Inicialização do Modelo

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False


inputs = layers.Input(shape=(256, 256, 3))


x = layers.Resizing(224, 224)(inputs)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
def validation_test_generator(X, y, batch_size):
    """Gera lotes para validação/teste com pré-processamento em tempo real."""
    num_samples = len(X)
    while True:
        for offset in range(0, num_samples, batch_size):
            X_batch = X[offset:offset+batch_size]
            y_batch = y[offset:offset+batch_size]

            X_processed_batch = tf.keras.applications.resnet50.preprocess_input(X_batch)

            yield (X_processed_batch, y_batch)

In [ ]:
X_val_processed = tf.keras.applications.resnet50.preprocess_input(X_val)

In [ ]:
BATCH_SIZE = 32

train_generator = balanced_data_generator(X_train, y_train, BATCH_SIZE)

In [ ]:
steps_per_epoch = math.ceil(len(X_train) / BATCH_SIZE)
validation_steps = math.ceil(len(X_val) / BATCH_SIZE)
test_steps = math.ceil(len(X_test) / BATCH_SIZE)

In [ ]:
checkpoint = ModelCheckpoint('best_model_resnet50_conv3_augmentation_90.keras', save_best_only=True, monitor='val_accuracy', mode='max')
early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

### Treinamento do Modelo

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=(X_val_processed, y_val),
    callbacks=[checkpoint, early_stopping]
)

In [ ]:
best_model = load_model('best_model_resnet50_conv3_augmentation_90.keras')

In [ ]:
X_test_processed = tf.keras.applications.resnet50.preprocess_input(X_test)
loss, accuracy = best_model.evaluate(X_test_processed, y_test)
print(f"Acurácia no teste: {accuracy:.2%}")

### Fine-Tuning

In [ ]:
try:
    model_for_finetuning = load_model('best_model_resnet50_conv3_augmentation_90.keras')
    print("Modelo carregado com sucesso para fine-tuning.")
    base_model_finetune = model_for_finetuning.get_layer('resnet50')

except Exception as e:
    print(f"Erro ao carregar o modelo 'best_model_resnet50_conv3_augmentation_90.keras': {e}")
    print("Verifique se o arquivo foi salvo corretamente na fase anterior.")

    base_model_finetune = model.get_layer('resnet50')

In [ ]:
base_model_finetune.trainable = True

fine_tune_at_layer_name = 'conv3_block1_out'
set_trainable = False
for layer in base_model_finetune.layers:
    if layer.name == fine_tune_at_layer_name:
        set_trainable = True
    if set_trainable:
        layer.trainable = True

    else:
        layer.trainable = False

print(f"\nNúmero total de camadas no base_model: {len(base_model_finetune.layers)}")
print(f"Número de camadas treináveis no base_model após descongelar: {len(base_model_finetune.trainable_weights) // 2}")

In [ ]:
model_for_finetuning.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_for_finetuning.summary()

### Treinamento Final do Modelo

In [ ]:
print("\nContinuando o treinamento (Fine-Tuning)...")

checkpoint_ft = ModelCheckpoint('fine_tuned_model_conv3_augmentation_90.keras', save_best_only=True, monitor='val_accuracy', mode='max')

early_stopping_ft = EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True)

reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', factor=0.2,
                            patience=3,
                            min_lr=1e-7,
                            verbose=1)

epochs_run_in_phase_1 = len(history.epoch)
fine_tune_epochs = 25

total_epochs = epochs_run_in_phase_1 + fine_tune_epochs
initial_epoch = epochs_run_in_phase_1

history_fine = model_for_finetuning.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=total_epochs,
    initial_epoch=initial_epoch,
    validation_data=(X_val_processed, y_val),
    callbacks=[checkpoint_ft, early_stopping_ft,reduce_lr]
)


In [ ]:
best_model = load_model('fine_tuned_model_conv3_augmentation_90.keras')

### Avaliação

In [ ]:
X_test_processed = tf.keras.applications.resnet50.preprocess_input(X_test)
loss, accuracy = best_model.evaluate(X_test_processed, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2%}")

Plotagem das curvas de aprendizado


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']


acc += history_fine.history['accuracy']
val_acc += history_fine.history['val_accuracy']
loss += history_fine.history['loss']
val_loss += history_fine.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação', marker='s', color='darkorange')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.grid(True)
plt.ylim([min(plt.ylim()), 1])

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino', marker='o', color='royalblue')
plt.plot(epochs_range, val_loss, label='Perda de Validação', marker='s', color='darkorange')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.grid(True)

plt.suptitle('Performance do Modelo ao Longo das Épocas', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plotagem da Matriz de Confusão

In [ ]:
y_pred_prob = best_model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test
cm = confusion_matrix(y_true, y_pred)

class_names = [
    "Disturbed Galaxy", "Merging Galaxy", "Round Smooth Galaxy",
    "In-between Round Smooth Galaxy", "Cigar Shaped Smooth Galaxy",
    "Barred Spiral Galaxy", "Unbarred Tight Spiral Galaxy",
    "Unbarred Loose Spiral Galaxy", "Edge-on Galaxy (No Bulge)",
    "Edge-on Galaxy (With Bulge)"
]

combined_labels = [f"{i} - {name}" for i, name in enumerate(class_names)]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=combined_labels,
    yticklabels=combined_labels,
    cbar=False,
    annot_kws={"fontsize": 9}
)

plt.xlabel("Classe Predita", fontsize=14)
plt.ylabel("Classe Verdadeira", fontsize=14)
plt.title("Matriz de Confusão - Teste", fontsize=16)


plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))